In [1]:
import pandas as pd
from google.cloud import bigquery

# Initialize client
client = bigquery.Client(project="knitwear-app")


def search_images(query_image_embedding: list[float], top_k: int = 50) -> pd.DataFrame:
    """Queries BigQuery for top visual matches using DINOv2 vector."""
    sql = """
    SELECT 
      base.pattern_id,
      distance AS image_distance
    FROM VECTOR_SEARCH(
      TABLE `knitwear-app.ravelry_data.dim_pattern_image_embeddings`,
      'image_embedding',
      (SELECT @query_vec AS image_embedding),
      top_k => @top_k,
      distance_type => 'COSINE'
    );
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("query_vec", "FLOAT64", query_image_embedding),
            bigquery.ScalarQueryParameter("top_k", "INT64", top_k),
        ]
    )
    return client.query(sql, job_config=job_config).to_dataframe()


def search_text(query_text: str, top_k: int = 50) -> pd.DataFrame:
    """Queries BigQuery for top semantic text matches via Vertex AI."""
    sql = """
    SELECT 
      base.pattern_id,
      base.original_text,
      distance AS text_distance
    FROM VECTOR_SEARCH(
      TABLE `knitwear-app.ravelry_data.dim_text_embeddings`,
      'text_embedding',
      (
        SELECT ml_generate_embedding_result 
        FROM ML.GENERATE_EMBEDDING(
          MODEL `knitwear-app.ravelry_data.pattern_text_embedder`,
          (SELECT @query_text AS content)
        )
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    );
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("query_text", "STRING", query_text),
            bigquery.ScalarQueryParameter("top_k", "INT64", top_k),
        ]
    )
    return client.query(sql, job_config=job_config).to_dataframe()


def late_fusion_search(
    query_image_vec: list[float] | None,
    query_text_str: str | None,
    alpha: float = 0.5,
    top_n: int = 10,
) -> pd.DataFrame:
    """
    Blends visual and semantic search results using Min-Max score normalization.
    
    alpha = 1.0 -> 100% Visual Search (DINOv2)
    alpha = 0.0 -> 100% Text Search (Vertex AI)
    alpha = 0.5 -> Equal Balance
    """
    # 1. Fetch candidate pools (Top 50 candidates from each modality)
    img_df = search_images(query_image_vec) if query_image_vec else pd.DataFrame()
    text_df = search_text(query_text_str) if query_text_str else pd.DataFrame()

    # Handle single-modality fallback
    if img_df.empty and not text_df.empty:
        return text_df.sort_values("text_distance").head(top_n)
    if text_df.empty and not img_df.empty:
        return img_df.sort_values("image_distance").head(top_n)

    # 2. Outer Join candidates on pattern_id
    merged = pd.merge(img_df, text_df, on="pattern_id", how="outer")

    # Fill missing scores with maximum observed distance (penalizes items found in only one modality)
    max_img_dist = merged["image_distance"].max() if not merged["image_distance"].isna().all() else 1.0
    max_txt_dist = merged["text_distance"].max() if not merged["text_distance"].isna().all() else 1.0
    
    merged["image_distance"] = merged["image_distance"].fillna(max_img_dist)
    merged["text_distance"] = merged["text_distance"].fillna(max_txt_dist)

    # 3. Min-Max Normalization (maps distances strictly to [0, 1])
    def min_max_scale(series: pd.Series) -> pd.Series:
        rng = series.max() - series.min()
        return (series - series.min()) / rng if rng > 0 else series * 0.0

    merged["norm_img_dist"] = min_max_scale(merged["image_distance"])
    merged["norm_txt_dist"] = min_max_scale(merged["text_distance"])

    # 4. Calculate Weighted Combined Distance
    # Lower combined score = higher overall relevance
    merged["combined_score"] = (alpha * merged["norm_img_dist"]) + ((1 - alpha) * merged["norm_txt_dist"])

    # 5. Return Top N ranked results
    results = merged.sort_values("combined_score").head(top_n)
    return results[["pattern_id", "combined_score", "image_distance", "text_distance", "original_text"]]

In [3]:
# Query both the ID and the embedding
sample_query = """
SELECT pattern_id, image_embedding 
FROM `knitwear-app.ravelry_data.dim_pattern_image_embeddings` 
LIMIT 1
"""

# Fetch the row
df_sample = client.query(sample_query).to_dataframe()

# Extract the pattern ID and the vector array
test_pattern_id = df_sample['pattern_id'].iloc[0]
sample_image_vector = df_sample['image_embedding'].iloc[0]

# Convert to list if it returns as a numpy array, just to be safe
if hasattr(sample_image_vector, "tolist"):
    sample_image_vector = sample_image_vector.tolist()

print(f"Go look up this pattern on your hard drive: {test_pattern_id}")

/opt/anaconda3/envs/knitwear_rec/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Go look up this pattern on your hard drive: 1000351


In [4]:
sample_text_query = "Cardigan with wavy pattern and shawl collar"

In [5]:

# Test 1: Equal weight (50% visual, 50% text)
balanced_results = late_fusion_search(
    query_image_vec=sample_image_vector,
    query_text_str=sample_text_query,
    alpha=0.5,
    top_n=5
)
print("--- Balanced Results (alpha = 0.5) ---")
print(balanced_results[["pattern_id", "combined_score", "image_distance", "text_distance"]])

# Test 2: Text-heavy weighting (80% text, 20% visual)
text_heavy_results = late_fusion_search(
    query_image_vec=sample_image_vector,
    query_text_str=sample_text_query,
    alpha=0.2,
    top_n=5
)
print("\n--- Text-Heavy Results (alpha = 0.2) ---")
print(text_heavy_results[["pattern_id", "combined_score", "image_distance", "text_distance"]])

/opt/anaconda3/envs/knitwear_rec/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


--- Balanced Results (alpha = 0.5) ---
    pattern_id  combined_score  image_distance  text_distance
97     7534749        0.500000         0.31297       0.259365
12     1000351        0.500000         0.00000       0.309302
78     7477080        0.534044         0.31297       0.262765
76     7473092        0.577307         0.31297       0.267086
90     7515922        0.683349         0.31297       0.277677


/opt/anaconda3/envs/knitwear_rec/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



--- Text-Heavy Results (alpha = 0.2) ---
    pattern_id  combined_score  image_distance  text_distance
97     7534749        0.200000         0.31297       0.259365
78     7477080        0.254471         0.31297       0.262765
76     7473092        0.323691         0.31297       0.267086
90     7515922        0.493359         0.31297       0.277677
2       198279        0.575610         0.31297       0.282811
